# Figures behind the `[viz]` extra

`axiom.viz` never imports plotly at import time. `available()` says whether the extra is
installed; every figure function returns a plotly `Figure` when it is and a typed
`Unsupported(reason="plotly not installed", missing=("viz",))` when it is not. Figures are
duck-typed over the result objects — each reads a documented set of attributes, so a missing
attribute is also a typed failure. Labels use the general vocabulary (treatment, dose,
outcome). Here we print figure sizes rather than rendering.

In [ ]:
import numpy as np

from axiom.core import Unsupported, clopper_pearson
from axiom.diagnose import (
    SpecificationAxis, SpecOption, coverage, rank_uniformity, rolling_origin, specification_curve,
)
from axiom.meta import forest_data, funnel_data, random_effects
from axiom.sim import DosePlan, surface_world
from axiom.surface import GeometricCarryover, fit
from axiom.viz import (
    available, backtest_plot, coverage_plot, forest, funnel, marginal_curve, response_curve,
    sbc_ranks, spec_curve_plot,
)

print("plotly available:", available())

## Response curve, forest, and funnel

`response_curve(result, treatment)` evaluates the fitted surface on a dose grid through
`predict_under` (one forward per draw) and draws the mean with a posterior band; `forest` and
`funnel` take `meta.forest_data` / `meta.funnel_data`.

In [ ]:
world = surface_world(n_units=2, n_periods=6, treatments=("a",), doses=DosePlan(scale=10.0), intercept="shared", seed=3)
res = fit(world.spec, world.panel, backend="laplace", draws=40, chains=1, seed=4)
fig = response_curve(res, "a", n_grid=8, mass=0.8)
print("response_curve traces:", len(fig.data), "| x:", fig.layout.xaxis.title.text, "| y:", fig.layout.yaxis.title.text)
# both surface figures render a surface.ResponseBand, so the band is the first trace and
# there is no argument on either that turns it off (tests/contracts/test_surface_uncertainty.py)
slope = marginal_curve(res, "a", n_grid=8, mass=0.8)
print("marginal_curve traces:", len(slope.data), "| band first:", slope.data[0].fill == "toself")

y, se = np.array([0.42, 0.55, 0.31, 0.67, 0.48]), np.array([0.10, 0.15, 0.12, 0.20, 0.11])
pooled = random_effects(y, se)
print("forest traces:", len(forest(forest_data(y, se, pooled, labels=list("abcde"))).data))
print("funnel traces:", len(funnel(funnel_data(y, se, pooled)).data))

## Diagnostic figures

`sbc_ranks` draws one histogram per `ParameterRanks`; `coverage_plot` shows each parameter's
rate against its acceptance region; `spec_curve_plot` sorts the rows of a `SpecCurve`;
`backtest_plot` shows the per-horizon scores of a `Backtest`. Each accepts any object with the
documented attributes — here real results from `axiom.diagnose` on tiny worlds.

In [ ]:
from types import SimpleNamespace

rng = np.random.default_rng(0)
sbc_like = SimpleNamespace(parameters=(
    rank_uniformity("beta_a", rng.integers(0, 20, size=60), n_ranks=19, alpha=0.05),
    rank_uniformity("k_a", np.minimum(rng.integers(0, 8, size=60), 19), n_ranks=19, alpha=0.05),
))
print("sbc_ranks traces:", len(sbc_ranks(sbc_like, columns=2).data))


def make_world(seed: int):
    return surface_world(n_units=3, n_periods=6, treatments=("a",), intercept="shared", noise_sd=0.3, seed=seed, doses=DosePlan(zero_fraction=0.2))


cov = coverage(make_world, n=4, draws=40, seed=1)
print("coverage_plot traces:", len(coverage_plot(cov).data))

In [ ]:
small = surface_world(n_units=3, n_periods=8, treatments=("a",), intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.2, seed=11)
axes = (SpecificationAxis(name="intercept_scale", options=(SpecOption(label="tight", spec_update={"intercept_scale": 0.5}), SpecOption(label="wide", spec_update={"intercept_scale": 5.0}))),)
curve = specification_curve(small.spec, small.panel, axes, draws=30, chains=1, seed=3)
print("spec_curve_plot traces:", len(spec_curve_plot(curve).data))

carry = surface_world(n_units=3, n_periods=10, treatments=("a",), carryover={"a": GeometricCarryover(max_lag=3)}, intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.1, seed=31)
bt = rolling_origin(carry.spec, carry.panel, origins=(7,), horizon=2, draws=30, chains=1, seed=0)
fig = backtest_plot(bt)
print("backtest_plot traces:", len(fig.data), "| json head:", fig.to_json()[:80])

## Typed failures

A result missing a documented attribute gives `Unsupported` naming what is missing; the same
happens for every figure when plotly is absent (the unit tests monkeypatch the import to
assert that path).

In [ ]:
out = coverage_plot(SimpleNamespace(mass=0.9, parameters=(SimpleNamespace(name="x"),)))
assert isinstance(out, Unsupported)
print(out.reason, "| missing:", out.missing)
print(backtest_plot(SimpleNamespace(mass=0.9)).missing)